In [1]:
from pymongo import MongoClient
import re
import os
from typing import List, Dict, Any
import shutil
import javalang

In [2]:
def connect_mongodb(uri,db_name, collection_name):
    # Connect to MongoDB 
    client = MongoClient(uri)
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]
    
    return collection
def get_all_ids(collection):
    return [doc['_id'] for doc in collection.find({}, {'_id': 1})]
    
def get_document_by_id(collection, target_id):
    return collection.find_one({"_id": target_id})
    
def get_level_3_path(data):
    level_2_path = data.get('level_2_path', [])
    level_3_values = []
    for item in level_2_path:
        level_3 = item.get('level_3_path', [])
        level_3_values.extend(level_3)
    return level_3_values

def get_files_with_prefix_and_extension(directory_path, prefix, extension, recursive=False):
    files = []

    if recursive:
        for root, dirs, filenames in os.walk(directory_path):
            for filename in filenames:
                if filename.startswith(prefix) and filename.endswith(extension):
                    files.append(os.path.join(root, filename))
    else:
        for filename in os.listdir(directory_path):
            if filename.startswith(prefix) and filename.endswith(extension):
                file_path = os.path.join(directory_path, filename)
                if os.path.isfile(file_path):
                    files.append(file_path)

    return files
def read_java_code_from_file(file_path: str) -> str:
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            code = file.read()
        return code
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return ""
    except Exception as e:
        print(f"Error reading file: {e}")
        return "" 
def remove_metadata_annotation(code):
    code = re.sub(r'@Metadata\s*\(.*?\)\s*', '', code, flags=re.DOTALL)
    return code

def clean_non_ascii_characters(code):
    return re.sub(r'[^\x00-\x7F]+', '', code)

def remove_string_literals(code):
    return re.sub(r'"(\\.|[^"\\])*"', '""', code)

def extract_ast_paths_2(code: str) -> List[str]:
    # Làm sạch code
    clean_code = remove_metadata_annotation(code)
    clean_code = clean_non_ascii_characters(clean_code)
    clean_code = remove_string_literals(clean_code)

    try:
        tree = javalang.parse.parse(clean_code)
    except (javalang.parser.JavaSyntaxError, javalang.tokenizer.LexerError) as e:
        # print(f"Error parsing Java code: {e}")
        return []
    except Exception as e:
        # print(f"Unexpected error: {e}")
        return []

    paths = []

    interested_node_types = (
        javalang.tree.Literal,
        javalang.tree.MemberReference,
        javalang.tree.MethodInvocation,
        javalang.tree.BinaryOperation,
        javalang.tree.ClassCreator,
        javalang.tree.FormalParameter,
        javalang.tree.VariableDeclarator,
    )

    def traverse(node, path):
        if isinstance(node, javalang.ast.Node):
            node_name = type(node).__name__
            path.append(node_name)
            if isinstance(node, interested_node_types):
                value = getattr(node, 'value', None) or getattr(node, 'member', None) or getattr(node, 'name', None)
                if value:
                    paths.append(".".join(path) + " -> " + str(value))
            for child in node.children:
                if isinstance(child, list):
                    for item in child:
                        traverse(item, path[:])
                elif child is not None:
                    traverse(child, path[:])

    traverse(tree, [])
    return paths

def remove_duplicates(lst):
    return list(dict.fromkeys(lst))
def get_filename_without_extension(file_path):
    filename = os.path.basename(file_path)
    return os.path.splitext(filename)[0]
def write_list_to_txt(file_path, data_list):
    with open(file_path, 'w', encoding='utf-8', errors='ignore') as f:
        for item in data_list:
            f.write(str(item) + '\n')

    # print(f"List has been written to {file_path}")

In [3]:
mongoDB_uri = 'mongodb://localhost:27017'
mongoDB_database = 'wearable-project-2' 
mongoDB_collection = 'codeblock_ast'

In [4]:
# Connect to the MongoDB collection
collection = connect_mongodb(mongoDB_uri,mongoDB_database,mongoDB_collection)

In [5]:
array_package_name = get_all_ids(collection)
print(len(array_package_name))

469


In [6]:
for x in range(len(array_package_name)):
    print("------------ Loop-"+str(x)+" ------------")
    package_name = array_package_name[x]
    print("package name: ",package_name)
    data = get_document_by_id(collection, package_name)
    # print("data: ",data)
    level_3_directory_array = get_level_3_path(data)
    # print("level_3_data: ",level_3_directory_array)
    for y in range(len(level_3_directory_array)):
        level_3_directory_path = level_3_directory_array[y]
        print("**********level_3_directory_path: ",level_3_directory_path)
        clear_java_file = get_files_with_prefix_and_extension(level_3_directory_path,"clean-",".java")
        if(len(clear_java_file)>0):
            for z in range(len(clear_java_file)):
                java_file_path = clear_java_file[z]
                # print(java_file_path)
                ast_file_name = get_filename_without_extension(java_file_path)
                java_code = read_java_code_from_file(java_file_path)
                ast = extract_ast_paths_2(java_code)
                final_ast = remove_duplicates(ast)
                ast_path = level_3_directory_path+"\\"+ast_file_name+".txt"
                write_list_to_txt(ast_path, final_ast)
        # break
    break

------------ Loop-0 ------------
package name:  adarshurs.android.vlcmobileremote
**********level_3_directory_path:  E:\wearable-2-dataset-code-extraction\adarshurs.android.vlcmobileremote\Google\com.google.android.gms.ads.MobileAds
**********level_3_directory_path:  E:\wearable-2-dataset-code-extraction\adarshurs.android.vlcmobileremote\Google\com.google.firebase.analytics.connector.AnalyticsConnector
**********level_3_directory_path:  E:\wearable-2-dataset-code-extraction\adarshurs.android.vlcmobileremote\Google\com.google.firebase.components.ComponentRegistrar
**********level_3_directory_path:  E:\wearable-2-dataset-code-extraction\adarshurs.android.vlcmobileremote\Google\com.google.firebase.provider.FirebaseInitProvider
